In [4]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00


In [11]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
import evaluate

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
EPOCHS = 1

# Load dataset
imdb = load_dataset("imdb")
train_dataset = imdb["train"]
test_dataset = imdb["test"]

# Optional: smaller subset for faster testing
# train_dataset = train_dataset.shuffle(seed=42).select(range(2000))
# test_dataset = test_dataset.shuffle(seed=42).select(range(1000))




In [12]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Keep only columns needed by the model
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir="./distilbert-imdb",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    report_to="none",
    disable_tqdm=True,   # important workaround for notebook callback bug
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print("Training finished.")



Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.498', 'grad_norm': '6.724', 'learning_rate': '1.873e-05', 'epoch': '0.06398'}
{'loss': '0.3189', 'grad_norm': '8.537', 'learning_rate': '1.745e-05', 'epoch': '0.128'}
{'loss': '0.3287', 'grad_norm': '11.55', 'learning_rate': '1.617e-05', 'epoch': '0.1919'}
{'loss': '0.2785', 'grad_norm': '9.765', 'learning_rate': '1.489e-05', 'epoch': '0.2559'}
{'loss': '0.2807', 'grad_norm': '10.06', 'learning_rate': '1.361e-05', 'epoch': '0.3199'}
{'loss': '0.2759', 'grad_norm': '6.225', 'learning_rate': '1.234e-05', 'epoch': '0.3839'}
{'loss': '0.2832', 'grad_norm': '7.274', 'learning_rate': '1.106e-05', 'epoch': '0.4479'}
{'loss': '0.2638', 'grad_norm': '1.161', 'learning_rate': '9.776e-06', 'epoch': '0.5118'}
{'loss': '0.2264', 'grad_norm': '9.876', 'learning_rate': '8.496e-06', 'epoch': '0.5758'}
{'loss': '0.278', 'grad_norm': '7.619', 'learning_rate': '7.217e-06', 'epoch': '0.6398'}
{'loss': '0.2568', 'grad_norm': '6.569', 'learning_rate': '5.937e-06', 'epoch': '0.7038'}
{'loss': '0.

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


{'train_runtime': '860.7', 'train_samples_per_second': '29.05', 'train_steps_per_second': '1.816', 'train_loss': '0.2827', 'epoch': '1'}
Training finished.


In [13]:
# This should now work
results = trainer.evaluate()
print("Evaluation results:", results)


{'eval_loss': '0.2295', 'eval_accuracy': '0.9101', 'eval_runtime': '209.5', 'eval_samples_per_second': '119.3', 'eval_steps_per_second': '7.462', 'epoch': '1'}
Evaluation results: {'eval_loss': 0.22953727841377258, 'eval_accuracy': 0.91008, 'eval_runtime': 209.4701, 'eval_samples_per_second': 119.349, 'eval_steps_per_second': 7.462, 'epoch': 1.0}


In [14]:
# Custom prediction
test_sentence = "This is a really good movie. I loved it and will watch again"

inputs = tokenizer(
    test_sentence,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_LENGTH,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

model.eval()
with torch.no_grad():
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)
    pred = torch.argmax(probs, dim=-1).item()

labels = ["Negative", "Positive"]
print("Prediction:", labels[pred])
print("Probabilities:", probs.cpu().numpy())

Prediction: Positive
Probabilities: [[0.00724872 0.99275124]]
